UCS420-assignment 4

In [9]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

In [10]:
#Q1
roll_no = input("Enter roll number: ")
cats = ["billing", "account", "general"]

extra = {
    "billing": [
        {"question": "how do i get a refund", "answer": "Refunds are processed within 7 working days.", "keywords": "refund money return"},
        {"question": "where can i download my invoice", "answer": "Open Billing > Invoices and click download.", "keywords": "invoice bill download"},
    ],
    "account": [
        {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Contact and verify the new number with an OTP.", "keywords": "mobile number update"},
        {"question": "how to delete my account", "answer": "Raise a request from Settings > Account > Delete.", "keywords": "delete close account"},
    ],
    "general": [
        {"question": "where is the campus located", "answer": "The campus is near the main bus stand.", "keywords": "campus location address"},
        {"question": "do you have a mobile app", "answer": "Yes, the app is on Play Store and App Store.", "keywords": "app download store"},
    ],
}

rows = list(fixed_entries)
for i, d in enumerate(roll_no[-2:]):
    cat = cats[int(d) % 3]
    entry = dict(extra[cat][i])
    entry["category"] = cat
    rows.append(entry)

df = pd.DataFrame(rows)
print(df)

Enter roll number: 1024170103
                          question  \
0           what is the annual fee   
1            how to reset password   
2      what are your working hours   
3            how can i pay the fee   
4            how do i get a refund   
5  where can i download my invoice   

                                         answer                keywords  \
0                     The annual fee is Rs 500.   fee cost price charge   
1              Go to Settings > Reset Password.    password reset login   
2                     We are open 9 AM to 5 PM.  hours timing open time   
3    You can pay via UPI, card, or net banking.     pay payment upi fee   
4  Refunds are processed within 7 working days.     refund money return   
5   Open Billing > Invoices and click download.   invoice bill download   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  billing  


In [11]:
#Q2
def score_query(query, df):
    words = query.lower().split()
    out = []
    if len(words) == 0:
        return pd.DataFrame(out, columns=["question", "answer", "confidence"])
    for i in range(len(df)):
        kw = df.loc[i, "keywords"].lower().split()
        qw = df.loc[i, "question"].lower().split()
        hits = [w for w in words if w in kw or w in qw]
        conf = len(hits) / len(words)
        if conf > 0:
            out.append((df.loc[i, "question"], df.loc[i, "answer"], round(conf, 2)))
    out.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(out, columns=["question", "answer", "confidence"])

print(score_query("pay fee", df))

                 question                                      answer  \
0   how can i pay the fee  You can pay via UPI, card, or net banking.   
1  what is the annual fee                   The annual fee is Rs 500.   

   confidence  
0         1.0  
1         0.5  


In [12]:
#Q3
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

cat = df["category"].iloc[-1]
print(cat)
print(same_category(cat, df))

billing
['what is the annual fee', 'how can i pay the fee', 'how do i get a refund', 'where can i download my invoice']


In [13]:
#Q4
new_kw = input("Enter a new keyword: ")
df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_kw
df.to_csv(roll_no + "_faq_data.csv", index=False)
print(df.loc[0])

Enter a new keyword: 5
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords      fee cost price charge 5
category                      billing
Name: 0, dtype: object


In [14]:
#Q5
print(df.groupby("category").size())

category
account    1
billing    4
general    1
dtype: int64


In [15]:
#Q6
def best_match(query, df):
    res = score_query(query, df)
    if res.empty:
        print("no match found")
        return
    top = res["confidence"].max()
    best = res[res["confidence"] == top]
    if len(best) > 1:
        print("tie, equally good matches:")
        print(best)
    else:
        print(best.iloc[0]["question"])
        print(best.iloc[0]["answer"])

best_match("fee", df)
print()
best_match("reset password", df)

tie, equally good matches:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

   confidence  
0         1.0  
1         1.0  

how to reset password
Go to Settings > Reset Password.
